In [1]:
import pandas as pd
from datetime import datetime, timedelta

### 计算工作时长的函数

In [2]:
def is_workday(date, holidays):
    # 判断是否为工作日（非周末且非节假日）
    return date not in holidays

def calculate_work_duration(start_time, end_time, holidays):
    # 计算工作时间（排除节假日和周末）
    current = start_time
    work_duration = timedelta()
    
    while current < end_time:
        next_day = (current + timedelta(days=1)).replace(hour=0, minute=0, second=0)
        if is_workday(current.date(), holidays):
            work_duration += min(next_day, end_time) - current
        current = next_day
    
    return work_duration.total_seconds() / (24 * 3600)  # 转换为天数

### 读取所需文件1、流程数据2、假期数据3、特殊节点数据

In [9]:
base_df = pd.read_excel(r"E:\000000我的事项\2025-09\1131统计\11月\审批数据-10月汇总-单节点.xlsx", engine='openpyxl')
holiday_df = pd.read_excel(r"E:\000000我的事项\2025-09\1131统计\11月\2025非工作日清单-截至1110.xlsx", engine='openpyxl')
special_node_df = pd.read_excel(r"E:\000000我的事项\2025-09\1131统计\11月\特殊节点时长基准-251105人力节点更新确认.xlsx",sheet_name='00', engine='openpyxl')
holidays = [datetime.strptime(str(date).strip(), '%Y-%m-%d %H:%M:%S').date() for date in holiday_df['非工作日日期']]

In [10]:
# 1. 计算自然时长和工作时长
base_df['单个节点审批到达时间'] = pd.to_datetime(base_df['单个节点审批到达时间'], format="mixed")
base_df['单个节点审批结束时间'] = pd.to_datetime(base_df['单个节点审批结束时间'], format="mixed")

# base_df['该节点审批自然时长'] = (base_df['单个节点审批结束时间'] - base_df['单个节点审批到达时间']).dt.total_seconds() / (24 * 3600)

base_df['该节点审批工作时长'] = base_df.apply(
    lambda row: calculate_work_duration(
        row['单个节点审批到达时间'], 
        row['单个节点审批结束时间'], 
        holidays
    ), 
    axis=1
)

# 2 规整工作时长（保留2位小数），小于0时设为0
base_df['该节点审批工作时长_规整'] = base_df['该节点审批工作时长'].apply(lambda x: max(0, round(x, 2)))

# 3.判断节点审批时效情况，按照1，2，3天分为3级
base_df['节点审批时效情况：≤1；1<X≤3；>3'] = base_df['该节点审批工作时长_规整'].apply(
    lambda x: '≤1' if x <= 1 else ('1<X≤3' if 1 < x <= 3 else  '>3')
)
base_df['流程节点拼接'] = base_df['流程名称']+base_df['审批节点名称']
# 4. 创建流程名称与建议时长的对照关系，依据特殊节点时长基准表
node_duration_map = dict(zip(
    special_node_df['流程节点拼接'],
    special_node_df['特殊合理时长基准（天）']
))

# 8. 匹配节点合理审批时长，默认值为1
base_df['节点审批时长'] = base_df['流程节点拼接'].map(lambda x: node_duration_map.get(x, 1))

# 9. 计算节点审批延期时长
base_df['节点审批延期时长(实际工作时长-节点审批时长）'] = base_df['该节点审批工作时长'] - base_df['节点审批时长']
base_df['节点审批延期时长(实际工作时长-节点审批时长）'] = base_df['节点审批延期时长(实际工作时长-节点审批时长）'].apply(lambda x: max(0, round(x, 2)))
# 10. 判断节点审批延期时长情况，按照1，2，3天分为3级
base_df['延期时长情况：0；≤1；1<X≤3；>3'] = base_df['节点审批延期时长(实际工作时长-节点审批时长）'].apply(
    lambda x: '0' if x == 0 else ('≤1' if x <= 1 else ('1<X≤3' if 1 < x <= 3 else '>3'))
)


In [11]:
base_df.to_excel(r'E:\000000我的事项\2025-09\1131统计\11月\审批结果.xlsx', index=False)

### 读取统计流程频次，节点数，时长所需数据文件

In [17]:
df_node = pd.read_excel(r"E:\000000我的事项\2025-09\1131统计\2-9月流程总时长和节点数分析-待统计.xlsx", sheet_name='计算节点数（不含拒绝、撤回）', engine='openpyxl')
df_process = pd.read_excel(r"E:\000000我的事项\2025-09\1131统计\2-9月流程总时长和节点数分析-待统计.xlsx", sheet_name='计算流程总时长（不含拒绝、撤回）', engine='openpyxl')

In [18]:
df_process['流程提交时间'] = pd.to_datetime(df_process['流程提交时间'],format="mixed")
df_process['流程结束时间'] = pd.to_datetime(df_process['流程结束时间'],format="mixed")
df_process['流程总时长（工作日）'] = df_process.apply(
    lambda row: calculate_work_duration(
        row['流程提交时间'], 
        row['流程结束时间'], 
        holidays
    ), 
    axis=1
)
from tqdm import tqdm
for index, row in tqdm(df_process.iterrows(), total=df_process.shape[0]):
    df_process.loc[index,'流程节点数'] = df_node[df_node['流程编号（每次提起后算1个编号）']==row['流程编号（每次提起后算1个编号）']].shape[0]
    df_process.loc[index,'串行流程节点数'] = df_node[df_node['流程编号（每次提起后算1个编号）']==row['流程编号（每次提起后算1个编号）']]['单个节点审批到达时间（格式如：2024-09-26 15:20:59，精确到秒）'].nunique()


In [19]:
df_process.to_excel(r"E:\000000我的事项\2025-09\1131统计\流程节点-时长数据.xlsx",index=False)

In [20]:
df_out1 = pd.DataFrame()
df_out1[['系统','流程名称']] = df_node[['所属IT系统','流程名称']].drop_duplicates().reset_index(drop=True)
df_out1[['频次','平均串行节点数','50分值-串行节点数','90分值--串行节点数','95分值--串行节点数','平均总时长','50分值-总时长','90分值-总时长','95分值-总时长','平均节点数','50分值-节点数','90分值--节点数','95分值--节点数']] = 0
for index,row in df_out1.iterrows():
    df_out1.loc[index,'频次'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程名称'].count()
    # df_out1.loc[index,'平均节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程节点数'].sum()/float(row['频次'])
    df_out1.loc[index,'50分值-节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程节点数'].quantile(0.5)
    df_out1.loc[index,'90分值--节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程节点数'].quantile(0.9)
    df_out1.loc[index,'95分值--节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程节点数'].quantile(0.95)
    # df_out1.loc[index,'平均总时长'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程总时长（工作日）'].sum()/float(row['频次'])
    df_out1.loc[index,'50分值-总时长'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程总时长（工作日）'].quantile(0.5)
    df_out1.loc[index,'90分值-总时长'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程总时长（工作日）'].quantile(0.9)
    df_out1.loc[index,'95分值-总时长'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程总时长（工作日）'].quantile(0.95)

    df_out1.loc[index,'50分值-串行节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['串行流程节点数'].quantile(0.5)
    df_out1.loc[index,'90分值--串行节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['串行流程节点数'].quantile(0.9)
    df_out1.loc[index,'95分值--串行节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['串行流程节点数'].quantile(0.95)


for index,row in df_out1.iterrows():
    df_out1.loc[index,'平均节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程节点数'].sum()/float(row['频次'])
    df_out1.loc[index,'平均串行节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['串行流程节点数'].sum()/float(row['频次'])  
    df_out1.loc[index,'平均总时长'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程总时长（工作日）'].sum()/float(row['频次'])
df_out1

C:\Users\zhangbon\AppData\Local\Temp\ipykernel_6276\4266439383.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5.599999999999998' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_out1.loc[index,'95分值--节点数'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程节点数'].quantile(0.95)
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_6276\4266439383.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '12.710405092592593' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_out1.loc[index,'50分值-总时长'] = df_process[(df_process['所属IT系统']==row['系统'])&(df_process['流程名称']==row['流程名称'])]['流程总时长（工作日）'].quantile(0.5)
C:\Users\zhangbon\AppData\Local\Temp\ipykernel_6276\4266439383.py:12: FutureWarning: Setting an item of incom

,系统,流程名称,频次,平均串行节点数,50分值-串行节点数,90分值--串行节点数,95分值--串行节点数,平均总时长,50分值-总时长,90分值-总时长,95分值-总时长,平均节点数,50分值-节点数,90分值--节点数,95分值--节点数
0,EHR,退休申请审批流程,15,3.533333,3.0,5.0,5.6,21.412704,12.710405,46.278877,59.392721,3.533333,3.0,5.0,5.6
1,EHR,补签审批流程,5101,1.129778,1.0,2.0,2.0,0.390765,0.014329,1.054086,1.985394,1.129778,1.0,2.0,2.0
2,EHR,请假审批流程,3368,1.379157,1.0,2.0,3.0,0.499911,0.047153,1.369520,2.268981,1.379157,1.0,2.0,3.0
3,EHR,公出审批流程,860,1.463953,1.0,2.0,3.0,0.706977,0.127234,1.984147,3.583124,1.463953,1.0,2.0,3.0
4,EHR,异动流程,148,7.878378,9.0,11.6,13.0,2.993111,2.561007,6.095817,8.178028,7.878378,9.0,11.6,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274,PLM,小批量通知流程,4,1.000000,1.0,1.0,1.0,160.979404,165.259803,214.407478,216.677969,1.000000,1.0,1.0,1.0
275,PLM,渠道退市资料收集签批流,1,1.000000,1.0,1.0,1.0,119.215394,119.215394,119.215394,119.215394,3.000000,3.0,3.0,3.0
276,PLM,OTS品质判断流程3,18,1.000000,1.0,1.0,1.0,105.990545,88.661412,115.999355,189.212282,1.000000,1.0,1.0,1.0
277,PLM,零部件检验标准签批流-海外,1,2.000000,2.0,2.0,2.0,213.724259,213.724259,213.724259,213.724259,3.000000,3.0,3.0,3.0


In [21]:
df_out1.to_excel(r"E:\000000我的事项\2025-09\1131统计\流程节点-时长数据-统计.xlsx",index=False)